In [15]:
# Transformer 는 encoder -> decoder
# encoder 를 이용해서 만든 언어모델 BERT : 감성분류, 스펨, 개체명인식, 유사도측정 --> 추출
# decoder 를 이용해서 만든 언어모델 GPT : 언어 추론 요약, QA 챗봇  --> 생성

In [16]:
# Bert가 잘하는 것 : 분류, 빈칸 추론, 문장 임베딩

In [17]:
!pip install 'git+https://github.com/SKTBrain/KoBERT.git#egg=kobert_tokenizer&subdirectory=kobert_hf'

ERROR: Invalid requirement: "'git+https://github.com/SKTBrain/KoBERT.git#egg=kobert_tokenizer": Expected package name at the start of dependency specifier
    'git+https://github.com/SKTBrain/KoBERT.git#egg=kobert_tokenizer
    ^
Hint: = is not a valid operator. Did you mean == ?
'subdirectory'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [18]:
from kobert_tokenizer import KoBERTTokenizer
tokenizer = KoBERTTokenizer.from_pretrained('skt/kobert-base-v1')

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.


In [19]:
from transformers import BertForMaskedLM
model = BertForMaskedLM.from_pretrained('skt/kobert-base-v1')
model.eval()

Some weights of BertForMaskedLM were not initialized from the model checkpoint at skt/kobert-base-v1 and are newly initialized: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(8002, 768, padding_idx=1)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwis

In [20]:
# 1. 상식 추론
import torch
text = '한국의 수도는 [MASK]입니다.'
inputs = tokenizer(text,return_tensors='pt')
mask_token_index = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)[1]
# 추론
with torch.no_grad():
  outputs = model(**inputs)
predictions = outputs.logits
print(predictions[0,mask_token_index,:])
masked_prediction = predictions[0,mask_token_index,:].topk(5)
for i, index_t in enumerate(masked_prediction.indices[0]):
  index = index_t.item()
  print(tokenizer.decode([index]))

tensor([[ 0.6533, -1.6825,  0.2657,  ..., -1.5248, -0.2595, -1.7162]])
빚
보이
딩
등
엇
